# Generador del archivo `corpus.json`

**Requisito:**

Este notebook lee `data/entidades.json` y `data/base_conocimiento.json` para redactar respuestas con datos reales (fechas, cuotas, bloques).

**Es importante que** los archivos .json de los códigos:
`02_generar_entidades.ipynb` y  `03_generar_base_conocimiento.ipynb`, existan previamente alojados en Github o se cuente con los respectivos archivos para subirlos al entorno en la carpeta `data` manualmente.


**Detalle:**
Se definen las intenciones cada una con su conjunto de utterances, junto con su respectivas respuestas predefinidas y guardadas finalmente en un archivo tipo .json `corpus.json` que se descargará al usuario.

#1) Celda de Importaciones

In [1]:
import json
import os

import shutil      # Copiar la carpeta data desde el repositorio de GitHub
import subprocess  # Ejecutar 'git clone' / 'git pull' para sincronizar data/ con GitHub

from google.colab import files  # Para descargar corpus.json, igual que en 02 y 03


#2) Sincronización con GitHub (traer entidades.json y base_conocimiento.json)

Debido a la limitacion de sesión los entornos se arrancan con disco vacio.

Esta celda crea la carpeta `data` para el presente entorno, se encarga de buscar los archivos json creados por el código `02` y `03` desde del repositorio Github, y en caso de que ocurra un fallo, permite al usario colocar los archivos json de forma manual al entorno y reejecutar el código para poder iniciar.

In [2]:
CARPETA_DATA = "data"

# URL del notebook principal.
URL_REPOSITORIO_GITHUB = "https://github.com/JohnnyMeta/PLN-Proyecto-parcial-2-GRUPO-PI-G2.git"


def sincronizar_carpeta_data(url_repositorio=URL_REPOSITORIO_GITHUB):
    # Asegurar que la carpeta 'data' exista en el entorno de ejecución
    if not os.path.exists(CARPETA_DATA):
        os.makedirs(CARPETA_DATA)
        print(f"Creada la carpeta local '{CARPETA_DATA}/' en el entorno.")

    nombre_repo = url_repositorio.rstrip("/").split("/")[-1].replace(".git", "")

    try:
        # Descargar o actualizar desde GitHub en segundo plano
        if not os.path.exists(nombre_repo):
            print(f"Clonando repositorio: {url_repositorio}")
            resultado = subprocess.run(
                ["git", "clone", url_repositorio],
                capture_output=True,
                text=True,
                timeout=30,
            )
        else:
            print("Repositorio ya clonado, actualizando con 'git pull'")
            resultado = subprocess.run(
                ["git", "-C", nombre_repo, "pull"],
                capture_output=True,
                text=True,
                timeout=30,
            )

        if resultado.returncode != 0:
            raise RuntimeError(resultado.stderr.strip())

        ruta_data_repo = os.path.join(nombre_repo, "data")
        if not os.path.isdir(ruta_data_repo):
            raise FileNotFoundError(
                f"No se encontró la carpeta 'data/' dentro de '{nombre_repo}'."
            )

        # Copiar y pegar el contenido
        for elemento in os.listdir(ruta_data_repo):
            ruta_origen = os.path.join(ruta_data_repo, elemento)
            ruta_destino = os.path.join(CARPETA_DATA, elemento)

            if os.path.isdir(ruta_origen):
                if os.path.exists(ruta_destino):
                    shutil.rmtree(ruta_destino)
                shutil.copytree(ruta_origen, ruta_destino)
            else:
                shutil.copy2(ruta_origen, ruta_destino)

        print("Contenido de data/ sincronizado desde GitHub.")

    except Exception as error:
        print(
            f"Aviso: No se pudo sincronizar con GitHub ({error}). Se usará el almacenamiento local."
        )

    # Leer el contenido final de la carpeta local e imprimir los archivos
    archivos_locales = os.listdir(CARPETA_DATA)
    if archivos_locales:
        print(
            f"Archivos actualmente disponibles en '{CARPETA_DATA}/': {archivos_locales}"
        )
    else:
        print(
            f"La carpeta '{CARPETA_DATA}/' está vacía. Si ocurrió un error de sincronización,"
        )
        print(
            "puedes copiar y pegar manualmente los archivos JSON dentro de ella utilizando el panel lateral de Colab."
        )


sincronizar_carpeta_data()

Creada la carpeta local 'data/' en el entorno.
Clonando repositorio: https://github.com/JohnnyMeta/PLN-Proyecto-parcial-2-GRUPO-PI-G2.git
Contenido de data/ sincronizado desde GitHub.
Archivos actualmente disponibles en 'data/': ['entidades.json', 'banner_ug.jpg', 'base_conocimiento.json']


#3) Carga de datos base (entidades y base de conocimiento)

Se leen ambos JSON generados previamente. Si no existen, se avisa de ello en vez de fallar

In [3]:
CARPETA_DATA = "data"
RUTA_ENTIDADES = os.path.join(CARPETA_DATA, "entidades.json")
RUTA_CONOCIMIENTO = os.path.join(CARPETA_DATA, "base_conocimiento.json")

def cargar_json_requerido(ruta, nombre_notebook_generador):
    if not os.path.exists(ruta):
        raise FileNotFoundError(
            f"No se encontró '{ruta}'. Ejecuta primero '{nombre_notebook_generador}' "
            "(o coloca el archivo en data/) antes de generar el corpus."
        )
    with open(ruta, mode="r", encoding="utf-8") as archivo:
        return json.load(archivo)

ENTIDADES = cargar_json_requerido(RUTA_ENTIDADES, "02_generar_entidades.ipynb")
BASE_CONOCIMIENTO = cargar_json_requerido(RUTA_CONOCIMIENTO, "03_generar_base_conocimiento.ipynb")
print("entidades.json y base_conocimiento.json cargados correctamente.")

entidades.json y base_conocimiento.json cargados correctamente.


#4) Nombre del archivo

In [4]:
ARCHIVO_SALIDA = "corpus.json"

#5) Utilidades — Plantilla de intención y formato de fechas

In [5]:
def crear_intencion(nombre, descripcion, utterances, responses):
    """Molde único para toda intención: garantiza que ninguna quede sin
    'descripcion', 'utterances' o 'responses' (los tres campos que
    cargar_corpus_desde_json() espera en el notebook principal)."""

    assert utterances, f"La intención '{nombre}' no puede tener 0 utterances."
    assert responses, f"La intención '{nombre}' no puede tener 0 respuestas."
    return {nombre: {"descripcion": descripcion, "utterances": utterances, "responses": responses}}


MESES_ES = {
    "01": "enero",
    "02": "febrero",
    "03": "marzo",
    "04": "abril",
    "05": "mayo",
    "06": "junio",
    "07": "julio",
    "08": "agosto",
    "09": "septiembre",
    "10": "octubre",
    "11": "noviembre",
    "12": "diciembre",
}

def formatear_fecha_larga(fecha_iso):
    """'2026-07-13' (formato página) -> 13 de julio de 2026"""
    anio, mes, dia = fecha_iso.split("-")
    return f"{int(dia)} de {MESES_ES[mes]} de {anio}"

def buscar_fecha_clave(id_fecha):
    return next(f for f in BASE_CONOCIMIENTO["fechas_clave"] if f["id"] == id_fecha)

#6) Intenciones Conversacionales Básicas

In [6]:
corpus_saludo = crear_intencion(
    "saludo",
    "Saludar al usuario e iniciar la conversación.",

    utterances=[
        # Saludos simples
        "hola",
        "buenos dias",
        "buenas tardes",
        "buenas noches",
        "buen dia",
        "hey",
        "hola que tal",
        "hola hola",
        "buenas",
        "saludos",

        # Inicio de conversación
        "hola necesito ayuda",
        "hola quisiera informacion",
        "hola necesito informacion",
        "quisiera hacer una consulta",
        "tengo una consulta",
        "quiero hacer una pregunta",
        "me puedes ayudar",
        "podrias ayudarme",
        "necesito ayuda",
        "tengo unas dudas",

        # Conversación informal
        "disculpa",
        "una pregunta",
        "consulta",
        "hola estas ahi",
        "hay alguien",
        "hola pe causa"

        ],

    responses=[
        "Bienvenido al asistente de admisiones de la Universidad de Guayaquil. Estoy aquí para ayudarte con información sobre carreras, requisitos, inscripción, cronogramas, cupos, nivelación y el proceso de admisión. ¿En qué puedo ayudarte?",
    ],
)

corpus_despedida = crear_intencion(
    "despedida",
    "Finalizar la conversación de forma cordial.",

    utterances=[
        # Despedidas
        "adios",
        "chao",
        "hasta luego",
        "nos vemos",
        "hasta pronto",
        "me despido",

        # Finalización
        "eso seria todo",
        "eso es todo",
        "ya no necesito nada",
        "ya no necesito mas",
        "no tengo mas preguntas",
        "no tengo mas dudas",
        "ya resolvi mi duda",

        # Agradecimiento + cierre
        "muchas gracias",
        "gracias eso es todo",
        "gracias por la ayuda",
        "gracias por la informacion",
        "muy amable",
        "perfecto gracias",

        # Informal
        "ok gracias",
        "listo gracias",
        "excelente gracias",
        "hasta la vista baby",
        "hasta la próxima",
        "hasta mañana",
        "hasta nunca",

        ],

    responses=[
        "¡Un gusto ayudarte! Exitos en tu proceso de admisión a la Universidad de Guayaquil.",
    ],
)

#7) Requisitos, Registro Nacional y Creación de Cuenta

In [7]:
corpus_requisitos = crear_intencion(
    "consultar_requisitos",
    "Consultar los requisitos generales para participar en el proceso de admisión.",

    utterances=[
        # Requisitos generales
        "que requisitos necesito",
        "cuales son los requisitos",
        "que necesito para ingresar",
        "que necesito para entrar",
        "que necesito para estudiar en la ug",
        "que piden para ingresar",
        "que debo presentar",
        "que debo tener para ingresar",
        "que necesito para postular",
        "requisitos para ingresar",
        "requisitos de admision",
        "requisitos para la universidad",
        "requisitos para estudiar",

        # Documentación
        "que documentos necesito",
        "que documentos debo presentar",
        "que papeles necesito",
        "que papeles debo entregar",
        "que documentos me piden",
        "documentos para ingresar",
        "documentos para la admision",

        # Lenguaje cotidiano
        "como puedo ingresar a la ug",
        "como entro a la universidad",
        "que necesito para ser estudiante",
        "que necesito para comenzar el proceso",
        "como inicio el proceso de admision",

        ],

    responses=[
        "Los requisitos principales para participar en el proceso de admisión son:\n\n"
        "1. Contar con el título de bachiller debidamente registrado u homologado.\n"
        "2. Tener cédula de identidad o pasaporte vigente.\n"
        "3. Haber completado el Registro Nacional en la plataforma del Ministerio de Educación (MINEDUC), requisito obligatorio antes de continuar con el proceso de admisión."
    ],
)


fecha_reg_nac = buscar_fecha_clave("REG_NAC")

corpus_registro_nacional = crear_intencion(
    "consultar_registro_nacional",
    "Informar sobre el Registro Nacional del MINEDUC.",

    utterances=[

        # Qué es
        "que es el registro nacional",
        "registro nacional",
        "en que consiste el registro nacional",
        "para que sirve el registro nacional",
        "porque debo hacer el registro nacional",

        # Cómo hacerlo
        "como hago el registro nacional",
        "como realizar el registro nacional",
        "como me registro en el mineduc",
        "como hago el registro en el ministerio",
        "donde hago el registro nacional",
        "donde debo registrarme",

        # Fechas
        "cuando es el registro nacional",
        "cuando empieza el registro nacional",
        "cuando termina el registro nacional",
        "fechas del registro nacional",
        "cronograma del registro nacional",

        # Consecuencias
        "es obligatorio el registro nacional",
        "que pasa si no hago el registro nacional",
        "puedo inscribirme sin el registro nacional",

        ],

    responses=[
        f"El Registro Nacional debe realizarse del "
        f"{formatear_fecha_larga(fecha_reg_nac['fecha_inicio'])} "
        f"al {formatear_fecha_larga(fecha_reg_nac['fecha_fin'])} "
        "en la plataforma del Ministerio de Educación (MINEDUC).\n\n"
        "Este registro es obligatorio para participar en el proceso de admisión de la Universidad de Guayaquil. "
        "Si no lo realizas dentro del período establecido, no podrás continuar con las siguientes etapas del proceso."
    ],
)



corpus_creacion_cuenta = crear_intencion(
    "explicar_creacion_cuenta",
    "Explicar cómo crear una cuenta en la plataforma institucional de admisiones.",

    utterances=[

        # Crear cuenta
        "como creo mi cuenta",
        "como crear una cuenta",
        "crear cuenta",
        "necesito crear una cuenta",
        "quiero crear una cuenta",
        "como hago mi cuenta",
        "como abrir una cuenta",

        # Usuario
        "como creo mi usuario",
        "como crear usuario",
        "donde creo mi usuario",
        "como obtengo un usuario",

        # Plataforma
        "como me registro en la plataforma",
        "como me registro en el sistema",
        "como entro al sistema de admision",
        "como accedo a la plataforma",
        "pasos para crear cuenta",

        # Consultas frecuentes
        "donde creo mi cuenta",
        "en que pagina creo mi cuenta",
        "como empiezo la inscripcion",
        "como inicio la inscripcion",
        "como comienzo el proceso de inscripcion",

        ],

    responses=[
        "Una vez realizado el Registro Nacional en la plataforma del MINEDUC, debes crear tu cuenta en la plataforma institucional de admisiones de la Universidad de Guayaquil.\n\n"
        "Desde esa cuenta podrás realizar tu inscripción y seguir las diferentes etapas del proceso de admisión, respetando el cronograma correspondiente al último dígito de tu cédula o pasaporte."
    ],
)

#8) Cronograma — Inscripción y Evaluación

In [8]:
cronograma_digitos = BASE_CONOCIMIENTO["cronograma_inscripcion_por_digito"]
lineas_cronograma = "; ".join(
    f"dígitos {c['digitos']} el {c['dia'].lower()} {formatear_fecha_larga(c['fecha'])}"
    if "fecha" in c
    else f"{c['digitos']} del {c['dia'].lower()} ({formatear_fecha_larga(c['fecha_inicio'])} al {formatear_fecha_larga(c['fecha_fin'])})"
    for c in cronograma_digitos
)

corpus_fechas_inscripcion = crear_intencion(
    "consultar_fechas_inscripcion",
    "Consultar el cronograma y fechas de inscripción.",

    utterances=[

        # Fechas generales
        "cuando son las inscripciones",
        "cuando empieza la inscripcion",
        "cuando inician las inscripciones",
        "cuando abren las inscripciones",
        "cuando puedo inscribirme",
        "cuando debo inscribirme",
        "cuando me toca inscribirme",
        "que dia me toca inscribirme",
        "que fecha me toca",
        "que fecha me corresponde",

        # Cronograma
        "cronograma de inscripcion",
        "cronograma de inscripciones",
        "fecha de inscripcion",
        "fechas de inscripcion",
        "calendario de inscripciones",

        # Último dígito
        "cuando me toca segun mi cedula",
        "cuando me toca segun el ultimo digito",
        "inscripcion por ultimo digito",
        "cronograma por cedula",

        # Fin del proceso
        "hasta cuando puedo inscribirme",
        "cuando terminan las inscripciones",
        "cuando cierran las inscripciones",

        ],

    responses=[
        f"La inscripción se realiza según el último dígito de tu cédula o pasaporte: {lineas_cronograma}"
    ],
)


fecha_cronograma_eval = buscar_fecha_clave("CRONOGRAMA_EVAL")
fecha_evaluacion = buscar_fecha_clave("EVALUACION")
corpus_cronograma_evaluacion = crear_intencion(
    "consultar_cronograma_evaluacion",
    "Consultar fechas del examen de admisión.",

    utterances=[

        # Examen
        "cuando es el examen",
        "cuando es el examen de admision",
        "fecha del examen",
        "fecha del examen de ingreso",
        "cuando tengo el examen",
        "cuando rindo el examen",
        "cuando doy el examen",
        "cuando me toca el examen",

        # Evaluación
        "cuando es la evaluacion",
        "cuando es la evaluacion de admision",
        "cuando me evaluan",
        "fecha de la evaluacion",

        # Cronograma
        "cuando publican el cronograma del examen",
        "cuando sale el horario del examen",
        "cuando publican el lugar del examen",
        "cuando sabre donde me toca rendir",
        "cuando dicen donde rendire",

        # Lenguaje cotidiano
        "cuando es la prueba",
        "cuando me hacen el examen",
        "cuando toca el examen",

        ],

    responses=[
        f"El cronograma de evaluaciones (fecha, hora y lugar de tu examen) se publica el {formatear_fecha_larga(fecha_cronograma_eval['fecha_inicio'])} en tu cuenta del sistema institucional. "
        f"Las evaluaciones se aplican entre el {formatear_fecha_larga(fecha_evaluacion['fecha_inicio'])} y el {formatear_fecha_larga(fecha_evaluacion['fecha_fin'])}, según el bloque de conocimiento de tu carrera. "
        "El examen representa el 50% del puntaje total de postulación."
    ],
)

#9) Bloques de Conocimiento y Carreras

In [9]:
def describir_materias(materias):
    return ", ".join(f"{m['materia']} {m['porcentaje']}% ({m['preguntas']} preguntas)" for m in materias)

nombres_bloques = "; ".join(
    f"Bloque {b['id'][-1]} ({b['nombre']})" for b in BASE_CONOCIMIENTO["bloques_conocimiento"]
)

corpus_bloques_conocimiento = crear_intencion(
    "explicar_bloques_conocimiento",
    "Explicar qué son los bloques de conocimiento.",

    utterances=[

        # Concepto
        "que es un bloque de conocimiento",
        "que son los bloques de conocimiento",
        "explicame los bloques",
        "explica los bloques de conocimiento",
        "que significa bloque de conocimiento",
        "para que sirven los bloques",
        "para que sirven los bloques de conocimiento",

        # Cantidad
        "cuantos bloques existen",
        "cuantos bloques hay",
        "cuantos son los bloques",

        # Organización
        "como se dividen las carreras",
        "como se agrupan las carreras",
        "como estan organizadas las carreras",
        "como funcionan los bloques",

        # Lenguaje cotidiano
        "que bloque hay",
        "bloques de la ug",
        "bloques ug",

        ],
    responses=[
        "Los bloques de conocimiento agrupan carreras afines a un área. "
        "El examen de admisión de la UG se organiza en 6 bloques:\n"
        f"{nombres_bloques}\n"
        "Si quieres el detalle de materias de un bloque puntual, indícame su número "
        "(por ejemplo: '¿qué materias tiene el bloque 3?').",
    ],
)

lineas_bloques_materias = "; ".join(
    f"Bloque {b['id'][-1]} ({b['nombre']}): {describir_materias(b['materias'])}"
    for b in BASE_CONOCIMIENTO["bloques_conocimiento"]
)


corpus_materias_bloque = crear_intencion(
    "consultar_materias_bloque",
    "Consultar qué materias evalúa un bloque de conocimiento específico.",
    utterances=[

        "que materias tiene el bloque",
        "que materias evalua el bloque",
        "que materias vienen",
        "materias del bloque",
        "contenido del bloque",
        "que viene en el bloque",

        "que entra en el examen del bloque",
        "que temas vienen",
        "que debo estudiar para el bloque",
        "que preguntan en el bloque",

        "cuantas preguntas tiene matematicas",
        "cuanto vale matematicas",
        "porcentaje de matematicas",

        "que porcentaje tiene quimica",

        ],
    responses=[
        "Cada bloque de conocimiento evalúa un conjunto de materias con distinto peso en el examen de admisión. "
        "Indícame el número de tu bloque (1 al 6)"
        "y te doy el desglose exacto de materias, porcentajes y número de preguntas.",
    ],
)



def nombres_carreras_por_bloque(bloque_id):
    return [c["nombre"] for c in ENTIDADES["carreras"].values() if c["bloque_id"] == bloque_id]

lineas_carreras_por_bloque = "; ".join(
    f"Bloque {b['id'][-1]} ({b['nombre']}): {', '.join(nombres_carreras_por_bloque(b['id']))}"
    for b in BASE_CONOCIMIENTO["bloques_conocimiento"]
)


corpus_carreras_bloque = crear_intencion(
    "consultar_carreras_bloque",
    "Consultar qué carreras pertenecen a cada bloque de conocimiento.",
    utterances=[
        "que carreras hay en el bloque",
        "que carreras pertenecen al bloque",
        "que carreras tiene el bloque",
        "que puedo estudiar en el bloque",
        "que carreras incluye",
        "que carreras entran",
        "que carreras estan en ese bloque",
        "carreras por bloque",
        "lista de carreras del bloque",
        "bloque carreras",

        ],
    responses=[
        "Puedo darte las carreras de un bloque específico. La UG agrupa sus carreras en 6 bloques de conocimiento (1 al 6). "
    ],
)

info_institucional = BASE_CONOCIMIENTO["informacion_institucional"]


_utterances_carrera_especifica_manual = [
    # Modelo de saludo
    "quiero estudiar medicina",
    "quiero estudiar derecho",
    "quiero estudiar arquitectura",
    "me interesa medicina",
    "me interesa software",
    "quiero ser medico",
    "quiero ser ingeniero",
    "quiero ser abogado",
    "informacion de medicina",
    "informacion sobre derecho",
    "como es medicina",
    "que tal medicina",
    "cuanto dura medicina",
    "que titulo da medicina",
    "modalidad de medicina",
]

_utterances_carrera_especifica_generadas = []
for _carrera in ENTIDADES["carreras"].values():
    _nombre_carrera = _carrera["nombre"].lower()
    _utterances_carrera_especifica_generadas.extend([
        f"quiero estudiar {_nombre_carrera}",
        f"informacion de {_nombre_carrera}",
        f"informacion sobre {_nombre_carrera}",
        f"me interesa {_nombre_carrera}",
        f"como es {_nombre_carrera}",
        f"cuanto dura {_nombre_carrera}",
    ])


# dict.fromkeys para mantener el orden ni introducir
# duplicados si alguna variante generada coincide con una manual.
_utterances_carrera_especifica = list(dict.fromkeys(
    _utterances_carrera_especifica_manual + _utterances_carrera_especifica_generadas
))

corpus_carrera_especifica = crear_intencion(
    "consultar_carrera_especifica",
    "Dar información detallada (título, duración, modalidad, sede) de una carrera puntual.",
    utterances=_utterances_carrera_especifica,
    responses=[
        f"Dime el nombre la carrera que te interesaría saber",
    ],
)



corpus_total_carreras = crear_intencion(
    "consultar_total_carreras",
    "Indicar el número total de carreras que ofrece la UG.",
    utterances=[
        "cuantas carreras hay",
        "cuantas carreras ofrece la ug",
        "cuantas carreras tiene la universidad",
        "cuantas carreras existen en la ug",
        "numero total de carreras",
        "cuantas carreras se pueden estudiar",
        "cuantas opciones hay",
        "cuantas profesiones hay",
        "cuantas carreras puedo estudiar",
        "cuantas carreras existen en total",

        ],
    responses=[
        f"La Universidad de Guayaquil ofrece un total de {info_institucional['total_carreras']} carreras, distribuidas en {info_institucional['total_facultades']} facultades y agrupadas en 6 bloques de conocimiento.",
    ],
)

corpus_total_facultades = crear_intencion(
    "consultar_total_facultades",
    "Indicar el número total de facultades de la UG y listarlas.",
    utterances=[
        "cuantas facultades hay",
        "cuantas facultades tiene la ug",
        "que facultades existen",
        "dime las facultades de la universidad",
        "cuantas facultades ofrece la universidad",

        ],
    responses=[
        "La UG cuenta con {} facultades: {}.".format(
            info_institucional["total_facultades"],
            ", ".join(f["nombre"] for f in ENTIDADES["facultades"]),
        ),
    ],
)

corpus_carreras_facultad = crear_intencion(
    "consultar_carreras_facultad",
    "Consultar qué carreras pertenecen a una facultad específica.",
    utterances=[
        "que carreras hay en la facultad de ",
        "carreras de la facultad de ",
        "que carreras pertenecen a la facultad de",
        "carreras que ofrece la facultad de",
        "que se puede estudiar en la facultad de",
        "que carreras tiene tal facultad",
        "cuantas facultades existen",
        "lista de facultades",
        "cuales son las facultades",
        "facultades disponibles",
        "que facultades tiene la universidad",

        ],
    responses=[
        "Puedo listarte las carreras de una facultad específica. Dime el nombre de la facultad "
    ],
)


#10) Cupos y Aceptación

In [10]:
corpus_asignacion_cupos = crear_intencion(
    "consultar_asignacion_cupos",
    "Explicar cómo se asignan los cupos.",

    utterances=[

        # Asignación
        "como se asignan los cupos",
        "como asignan los cupos",
        "como funciona la asignacion de cupos",
        "como reparten los cupos",
        "como entregan los cupos",
        "como deciden quien obtiene un cupo",

        # Criterios
        "como eligen quien entra",
        "como eligen quien entra a la carrera",
        "de que depende obtener un cupo",
        "que toman en cuenta para dar un cupo",
        "cuales son los criterios para obtener un cupo",
        "criterios para asignar cupo",

        # Puntaje
        "el puntaje influye en el cupo",
        "como influye mi nota",
        "como influye el examen",
        "como calculan el puntaje final",

        # Empates
        "que pasa si hay empate",
        "que pasa si dos personas tienen el mismo puntaje",
        "como resuelven un empate",
        "que pasa si empato por el ultimo cupo",

        ],

    responses=[
        "El cupo se asigna según el orden de mérito (50% examen de admisión + 50% antecedentes académicos, más acciones afirmativas si aplican). "
        "En caso de empate para el último cupo disponible, se considera la nota del registro académico, las políticas de acción afirmativa y el tiempo empleado en la evaluación del campo específico de conocimiento."
    ],
)

lineas_cuotas = "; ".join(f"{c['grupo']}: {c['porcentaje']}%" for c in BASE_CONOCIMIENTO["cuotas_admision"])
corpus_cuotas_admision = crear_intencion(
    "consultar_cuotas_admision",
    "Consultar las cuotas de admisión y acción afirmativa.",

    utterances=[

        # Acción afirmativa
        "que es la accion afirmativa",
        "como funciona la accion afirmativa",
        "para que sirve la accion afirmativa",

        # Grupos
        "hay cupos para grupos vulnerables",
        "hay cupos especiales",
        "hay cupos preferenciales",
        "hay cupos para bachilleres",
        "que grupos tienen cupos",

        # Distribución
        "como se reparten los cupos",
        "como se distribuyen los cupos",
        "porcentaje de cupos",
        "cuotas de admision",
        "cupos por grupo",

        # Mérito
        "cuanto cupo hay para merito academico",
        "como se divide la oferta de cupos",

        ],

    responses=[
        f"La oferta de cupos por carrera se distribuye mediante cuotas de admisión y grupos de acción afirmativa. La distribución es la siguiente: {lineas_cuotas}."
    ],
)


corpus_aceptacion_cupo = crear_intencion(
    "consultar_aceptacion_cupo",
    "Explicar el proceso de aceptación del cupo.",

    utterances=[

        # Aceptación
        "como acepto mi cupo",
        "como aceptar el cupo",
        "donde acepto mi cupo",
        "tengo que aceptar el cupo",
        "es obligatorio aceptar el cupo",

        # Después del cupo
        "que pasa si acepto el cupo",
        "que hago despues de obtener un cupo",
        "que sigue despues del cupo",
        "que debo hacer si obtuve un cupo",

        # Cambios
        "puedo cambiar de carrera despues de aceptar",
        "puedo cambiar el cupo",
        "puedo anular mi cupo",
        "puedo cancelar mi cupo",
        "puedo renunciar al cupo",

        # Consecuencias
        "que pasa si no acepto el cupo",
        "pierdo el cupo si no acepto",
        "puedo aceptar despues",

        ],

    responses=[
        "La aceptación del cupo es un acto libre y voluntario que realizas en la plataforma del MINEDUC. "
        "Una vez aceptado, el cupo es definitivo: no puede modificarse ni anularse y debe utilizarse únicamente en el período correspondiente. "
        "El proceso contempla tres etapas de postulación y aceptación (primera, segunda y tercera), todas voluntarias."
    ],
)

#11) Información Institucional

In [11]:
razones = " ".join(f"({i+1}) {r}" for i, r in enumerate(info_institucional["razones_estudiar_ug"]))
corpus_por_que_ug = crear_intencion(
    "consultar_por_que_estudiar_ug",
    "Explicar las principales razones para estudiar en la Universidad de Guayaquil.",

    utterances=[

        # Razones
        "por que estudiar en la ug",
        "por que estudiar en la universidad de guayaquil",
        "por que elegir la ug",
        "por que elegir la universidad de guayaquil",
        "dame razones para estudiar en la ug",
        "dame razones para elegir la ug",

        # Ventajas
        "que ventajas tiene la ug",
        "que ventajas tiene la universidad",
        "que beneficios tiene estudiar en la ug",
        "cuales son las ventajas de estudiar en la ug",
        "que ofrece la universidad de guayaquil",

        # Opinión
        "vale la pena estudiar en la ug",
        "es buena la universidad de guayaquil",
        "es recomendable estudiar en la ug",
        "por que deberia estudiar en la ug",

        # Lenguaje cotidiano
        "que tiene de bueno la ug",
        "que hace diferente a la ug",
        "por que me conviene estudiar aqui",
        "por que escoger la ug",

        ],

    responses=[
        f"Estas son algunas de las principales razones para estudiar en la Universidad de Guayaquil:\n\n{razones}"
    ],
)

corpus_fuente_oficial = crear_intencion(
    "consultar_fuente_oficial",
    "Informar cuáles son los canales oficiales de la Universidad de Guayaquil.",

    utterances=[

        # Página oficial
        "donde consigo informacion oficial",
        "donde encuentro informacion oficial",
        "cual es la pagina oficial",
        "cual es la pagina oficial de admisiones",
        "pagina oficial de la ug",
        "sitio web oficial",

        # Redes
        "hay redes sociales oficiales",
        "cuales son las redes oficiales",
        "facebook oficial",
        "instagram oficial",
        "redes sociales de la universidad",

        # Seguridad
        "como evito cuentas falsas",
        "como saber si una pagina es oficial",
        "como identificar informacion oficial",
        "donde verifico la informacion",

        # General
        "donde puedo informarme",
        "donde puedo consultar informacion",
        "donde reviso las novedades",
        "donde publican los comunicados",
        "donde publican las fechas",

        ],

    responses=[
        f"Toda la información oficial del proceso de admisión se publica en los canales institucionales de la Universidad de Guayaquil: {info_institucional['fuente_oficial']}. {info_institucional['advertencia_fuentes']}"
    ],
)

#12) Curso de nivelación de carrera

In [12]:
CURSO_NIVELACION = BASE_CONOCIMIENTO["curso_nivelacion"]
_req_niv = CURSO_NIVELACION["requisitos_matricula"]
_eval_niv = CURSO_NIVELACION["evaluacion"]
_aprob_niv = CURSO_NIVELACION["aprobacion"]
_plat_niv = CURSO_NIVELACION["plataformas_virtuales"]

corpus_explicar_nivelacion = crear_intencion(
    "explicar_curso_nivelacion",
    "Explicar qué es el Curso de Nivelación.",

    utterances=[

        # Concepto
        "que es el curso de nivelacion",
        "que es la nivelacion",
        "que es la nivelacion de carrera",
        "en que consiste la nivelacion",
        "para que sirve la nivelacion",
        "para que sirve el curso de nivelacion",

        # Obligatoriedad
        "es obligatorio el curso de nivelacion",
        "tengo que hacer la nivelacion",
        "debo hacer la nivelacion",
        "es necesario hacer la nivelacion",

        # Después del cupo
        "que pasa despues de obtener un cupo",
        "que sigue despues del cupo",
        "que hago despues del cupo",
        "que sigue despues de ser admitido",

        # Lenguaje cotidiano
        "explicame la nivelacion",
        "como funciona la nivelacion",
        "quiero saber sobre la nivelacion",

        ],

    responses=[
        CURSO_NIVELACION["descripcion"]
    ],
)

_documentos_niv = "; ".join(f"{i+1}) {d}" for i, d in enumerate(_req_niv["documentos"]))
_pasos_niv = " ".join(f"({i+1}) {p}" for i, p in enumerate(_req_niv["pasos_adicionales"]))
corpus_requisitos_matricula_nivelacion = crear_intencion(
    "consultar_requisitos_matricula_nivelacion",
    "Consultar requisitos y documentos para matricularse en la nivelación.",
    utterances=[
        "como me matriculo",
        "como hago la matricula",
        "que necesito para matricularme",
        "documentos para matricularme",
        "que documentos debo subir",
        "que archivos debo subir",
        "que papeles necesito",
        "como subo los documentos",
        "que formato deben tener",
        "matricula de nivelacion",
        "requisitos de matricula",
        ],
    responses=[
        f"{_req_niv['modalidad']} Los documentos a adjuntar (escaneados a color) son: {_documentos_niv}. {_req_niv['formato_documentos']} Además debes: {_pasos_niv}",
    ],
)

corpus_evaluacion_aprobacion_nivelacion = crear_intencion(
    "consultar_evaluacion_aprobacion_nivelacion",
    "Explicar cómo se evalúa y qué se requiere para aprobar la nivelación.",
    utterances=[
        "como apruebo",
        "que nota necesito",
        "con cuanto apruebo",
        "como se evalua",
        "como califican",
        "que pasa si repruebo",
        "pierdo la nivelacion",
        "cuanto debo sacar",
        "cuanta asistencia necesito",
        "como funciona la evaluacion",
        ],
    responses=[
        "La evaluación se compone de: "
        + "; ".join(f"{c['criterio']} ({c['porcentaje']}%)" for c in _eval_niv["componentes"])
        + f". {_aprob_niv['descripcion']}",
    ],
)


corpus_plataformas_virtuales_nivelacion = crear_intencion(
    "consultar_plataformas_virtuales_nivelacion",
    "Informar sobre el Campus Virtual (Moodle) y el Aula Virtual (Zoom).",
    utterances=[
        "donde son las clases",
        "donde entro",
        "como entro al campus virtual",
        "como ingreso al moodle",
        "como ingreso al aula virtual",
        "donde veo las clases",
        "donde veo las tareas",
        "donde estan las grabaciones",
        "como entro a zoom",
        "usuario del campus virtual",
        "credenciales del campus",
        ],

    responses=[
        f"{_plat_niv['campus_virtual']['nombre']}: {_plat_niv['campus_virtual']['descripcion']} "
        f"Enlace: {_plat_niv['campus_virtual']['url']}. {_plat_niv['campus_virtual']['primer_ingreso']} "
        f"{_plat_niv['campus_virtual']['disponibilidad']} "
        f"{_plat_niv['aula_virtual']['nombre']}: {_plat_niv['aula_virtual']['descripcion']} "
        f"Ruta de acceso en el SIUG: {_plat_niv['aula_virtual']['ruta_acceso_siug']}. "
        f"{_plat_niv['aula_virtual']['donde_ver_grabaciones']}",
    ],
)

fecha_inicio_clases_niv = buscar_fecha_clave("INICIO_CLASES_NIV")
fecha_retiro_definitivo_niv = buscar_fecha_clave("RETIRO_DEFINITIVO_NIV")
fecha_registro_retiro_niv = buscar_fecha_clave("REGISTRO_SOLICITUD_RETIRO_NIV")
fecha_max_aprobacion_retiro_niv = buscar_fecha_clave("FECHA_MAX_APROBACION_RETIRO_NIV")
corpus_fechas_nivelacion = crear_intencion(
    "consultar_fechas_nivelacion", "Consultar fechas de inicio de clases y de retiro del curso de nivelación.",
    utterances=[
        "cuando empiezan las clases de nivelacion",
        "cuando inician las clases",
        "cuando comienzan las clases",
        "fecha de inicio",
        "cuando empieza la nivelacion",

        "hasta cuando puedo retirarme del curso de nivelacion",
        "hasta cuando puedo retirarme",
        "fecha de retiro",
        "cuando puedo retirarme",
        "cuando es el retiro definitivo",
        "cuando termina el plazo de retiro",
        "fecha limite para retirar por caso fortuito",
        "cuando puedo pedir el retiro por fuerza mayor",

        ],
    responses=[
        f"Las clases del Curso de Nivelación inician el {formatear_fecha_larga(fecha_inicio_clases_niv['fecha_inicio'])}. "
        f"El retiro definitivo se puede solicitar del {formatear_fecha_larga(fecha_retiro_definitivo_niv['fecha_inicio'])} "
        f"al {formatear_fecha_larga(fecha_retiro_definitivo_niv['fecha_fin'])}. Las solicitudes de retiro por caso "
        f"fortuito o fuerza mayor se registran del {formatear_fecha_larga(fecha_registro_retiro_niv['fecha_inicio'])} "
        f"al {formatear_fecha_larga(fecha_registro_retiro_niv['fecha_fin'])}, y deben quedar aprobadas a más tardar "
        f"el {formatear_fecha_larga(fecha_max_aprobacion_retiro_niv['fecha_inicio'])}.",
    ],
)

corpus_creacion_cuenta_siug = crear_intencion(
    "explicar_creacion_cuenta_siug",
    "Explicar la creación/uso de cuenta en el SIUG para la matrícula de nivelación.",
    utterances=[

        "como creo mi cuenta en el siug",
        "necesito cuenta en el siug",
        "ya tengo cuenta",
        "ya tenia cuenta",
        "debo crear otra cuenta",
        "uso la misma cuenta",
        "como ingreso al siug",
        "como me registro en el siug",
        "necesito una cuenta nueva",

        ],
    responses=[_req_niv["nota_cuenta_existente"]],
)

#13) Agrupar

In [13]:
corpus = {
    # ** - Alternativa a corpus.update
    **corpus_saludo,
    **corpus_despedida,
    **corpus_requisitos,
    **corpus_registro_nacional,
    **corpus_creacion_cuenta,
    **corpus_fechas_inscripcion,
    **corpus_cronograma_evaluacion,
    **corpus_bloques_conocimiento,
    **corpus_carreras_bloque,
    **corpus_carrera_especifica,
    **corpus_materias_bloque,
    **corpus_total_carreras,
    **corpus_total_facultades,
    **corpus_carreras_facultad,
    **corpus_asignacion_cupos,
    **corpus_cuotas_admision,
    **corpus_aceptacion_cupo,
    **corpus_por_que_ug,
    **corpus_fuente_oficial,
    **corpus_explicar_nivelacion,
    **corpus_requisitos_matricula_nivelacion,
    **corpus_evaluacion_aprobacion_nivelacion,
    **corpus_plataformas_virtuales_nivelacion,
    **corpus_fechas_nivelacion,
    **corpus_creacion_cuenta_siug,
}

#14) Validación

In [14]:
print(f"Total de intenciones: {len(corpus)}")
assert len(corpus) >= 10, "El RF-01 exige un mínimo de 10 intenciones principales."

todas_las_utterances = []
for nombre, datos in corpus.items():
    print(f"{nombre}: {len(datos['utterances'])} frases, {len(datos['responses'])} respuestas")
    todas_las_utterances.extend(datos["utterances"])

duplicadas = {u for u in todas_las_utterances if todas_las_utterances.count(u) > 1}
if duplicadas:
    print(f"\nAviso: hay utterances repetidas entre intenciones distintas: {duplicadas}")
else:
    print("\nSin utterances duplicadas entre intenciones. Total de utterances:", len(todas_las_utterances))

Total de intenciones: 25
saludo: 26 frases, 1 respuestas
despedida: 26 frases, 1 respuestas
consultar_requisitos: 25 frases, 1 respuestas
consultar_registro_nacional: 19 frases, 1 respuestas
explicar_creacion_cuenta: 21 frases, 1 respuestas
consultar_fechas_inscripcion: 22 frases, 1 respuestas
consultar_cronograma_evaluacion: 20 frases, 1 respuestas
explicar_bloques_conocimiento: 17 frases, 1 respuestas
consultar_carreras_bloque: 10 frases, 1 respuestas
consultar_carrera_especifica: 361 frases, 1 respuestas
consultar_materias_bloque: 14 frases, 1 respuestas
consultar_total_carreras: 10 frases, 1 respuestas
consultar_total_facultades: 5 frases, 1 respuestas
consultar_carreras_facultad: 11 frases, 1 respuestas
consultar_asignacion_cupos: 20 frases, 1 respuestas
consultar_cuotas_admision: 15 frases, 1 respuestas
consultar_aceptacion_cupo: 17 frases, 1 respuestas
consultar_por_que_estudiar_ug: 19 frases, 1 respuestas
consultar_fuente_oficial: 20 frases, 1 respuestas
explicar_curso_nivelaci

#15) Guardar el archivo

In [15]:
with open(ARCHIVO_SALIDA, "w", encoding="utf-8") as archivo:
    json.dump(corpus, archivo, indent=4, ensure_ascii=False)

# Descarga de archivo
files.download(ARCHIVO_SALIDA)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>